# Welcome back!

In Notebook 1, you stepped in as a D ONE Consultant to save D ONE's data operations from complete collapse. You successfully landed raw client feeds into Unity Catalog Volumes, audited corrupt records, and engineered pristine Bronze and Silver Medallion layers.

Now, the client’s executive leadership and BI teams are asking for an analytical reporting layer. They don't want to query flat, cleansed tables directly—they need a structured Star Schema optimized for high-performance dashboarding (PowerBI, Tableau, Databricks SQL) and Slowly Changing Dimensions (SCD Type 2) to track consultant promotions and practice transfers accurately over time.


<pre>
┌───────────────────────────────────────────────────────────────────────────────┐
│                        STAR SCHEMA ARCHITECTURE                               │
└───────────────────────────────────────────────────────────────────────────────┘
                                      │
             ┌────────────────────────┼────────────────────────┐
             ▼                        ▼                        ▼
     ┌───────────────┐        ┌───────────────┐        ┌───────────────┐
     │ DimConsultant │        │ FactTimesheet │        │  DimProject   │
     │  (SCD Type 2) │        │  (Fact Table) │        │  (SCD Type 1) │
     └───────────────┘        └───────────────┘        └───────────────┘
</pre>

## Step 0: Environment & Context Setup

We start by reusing parameters from Workshop 1 to maintain catalog and schema context.

In [0]:
%run ../___setup

In [0]:
# Set catalog and schema context
spark.sql(f"USE CATALOG `{CATALOG_NAME}`")
spark.sql(f"USE SCHEMA `{SCHEMA_NAME}`")

display(spark.sql("SELECT current_catalog(), current_schema()"))

##📚 Theory Break #1: What is a Star Schema?

Before building tables, let's establish the architectural fundamentals:

### Why not query Silver tables directly?
Silver tables are normalized operational entities. Joining multiple flat Silver tables during live executive reporting causes massive CPU overhead, slow dashboard load times, and complex SQL logic for end users.  

### The Star Schema Solution

A Star Schema organizes data into two specialized table categories:

- **Fact Tables:** Located at the center of the star. They store quantitative, numeric measurement data (e.g., hours logged, revenue, costs) alongside foreign surrogate keys linking to dimension tables. - Fact Tables: Located at the center of the star. _Example: fact_timesheet_

- **Dimension Tables:** The "points" of the star. They contain descriptive attributes that provide context to facts (e.g., consultant names, practice areas, project budgets). _Examples: dim_consultant, dim_project_


## Step 1: Model the Star Schema Dimensions & Facts

### 🧩 Hands-On Task 1.1: Build `dim_project` (Type 1 Dimension)
> **Scenario**: Create dim_project as a dimension sourced from silver_projects

In [0]:
load_hint("notebook_2", "step_1.1")

In [0]:
# TASK 1.1: Fill in the source table name to build dim_project
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`dim_project` AS
SELECT 
    project_hash_key,
    project_code,
    project_name,
    industry,
    budget
FROM ...........
""")

display(spark.sql(f"SELECT * FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`dim_project`"))

### 🧩 Hands-On Task 1.2: Build `fact_timesheet` Fact Table

> Complete the SQL query to join `silver_timesheets` with `silver_consultants` and `silver_projects` to populate foreign hash keys and facts (hours).

In [0]:
load_hint("notebook_2", "step_1.2")

In [0]:
# TASK 1.2: Complete the join ON conditions to build fact_timesheet
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`fact_timesheet` AS
SELECT 
    t.timesheet_hash_key,
    c.consultant_hash_key,
    p.project_hash_key,
    t.work_date,
    t.hours
FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_timesheets` t
JOIN `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_consultants` c 
    ON .............
JOIN `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_projects` p 
    ON .............
""")

display(spark.sql(f"SELECT * FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`fact_timesheet`"))

## 📚 Theory Break #2: Slowly Changing Dimensions (SCD) & Delta Save Modes

###What is Slowly Changing Dimensions (SCD)?
Attributes in dimension tables change over time. How we manage these changes determines our reporting accuracy:

- SCD Type 0 (Retain Original): Attributes are locked permanently on insertion (e.g., original_hire_date).

- SCD Type 1 (Overwrite): Replaces old data with new data directly. Historical values are lost.

- SCD Type 2 (Add New Row - Full History): Tracks full historical changes by inserting a new row with updated values, marking previous records as inactive using validity flags (is_current, valid_from, valid_to).


<pre>
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                              SCD TYPE 2 TRACKING SAMPLE                                │
├───────────────┬─────────────┬──────────────────┬────────────┬────────────┬─────────────┤
│ consultant_id │  full_name  │ seniority_level  │ is_current │ valid_from │  valid_to   │
├───────────────┼─────────────┼──────────────────┼────────────┼────────────┼─────────────┤
│     C001      │ Elena Rossi │ Senior           │   false    │ 2026-01-01 │ 2026-09-15  │
│     C001      │ Elena Rossi │ Lead             │   true     │ 2026-09-15 │    NULL     │
└───────────────┴─────────────┴──────────────────┴────────────┴────────────┴─────────────┤
</pre>

## Step 2: Implement Slowly Changing Dimensions (SCD Type 2)

%md
### 🧩 Hands-On Task 2.1: Initialize dim_consultant with SCD Tracking Flags

When a consultant gets promoted or changes practices, we don't overwrite their record—we keep history by marking the old record as inactive (is_current = false) and creating a new record (is_current = true). We start by building dim_consultant from silver_consultants, adding validity tracking attributes (is_current, valid_from, valid_to).

> You only need to replace the ...... placeholders.

In [0]:
load_hint("notebook_2", "step_2.1")

In [0]:
# TASK 2.1: Fill in initial tracking flags for SCD Type 2 setup
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`dim_consultant` AS
SELECT 
    consultant_hash_key,
    consultant_id,
    full_name,
    practice,
    seniority_level,
    -- Set active flag and validity dates
    ............. AS is_current,
    DATE('2026-01-01') AS valid_from,
    CAST(............. AS DATE) AS valid_to
FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_consultants`
""")

display(spark.sql(f"SELECT * FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`dim_consultant`"))

### 🔍 How MERGE INTO Works in SCD Type 2 (The UNION ALL Technique)

In **SCD Type 1**, the `MERGE INTO` operation simply performs an `UPDATE` when a match is found (overwrite).  
In **SCD Type 2**, when an attribute changes (e.g., a consultant promotion), two actions must occur simultaneously:
1. **Invalidation (Update):** The existing active record (`is_current = TRUE`) must be updated to `is_current = FALSE` with an updated `valid_to` date.
2. **Insertion (Insert):** A **new** row must be created with the updated values (`is_current = TRUE`).

**Why do we use `UNION ALL` inside `USING (src)`?**
For every consultant record being updated, we generate **2 rows in `src`**:
- **The 1st row** retains the actual `consultant_id` as the `merge_key` -> Matches the `WHEN MATCHED` condition and **expires (updates)** the historical record.
- **The 2nd row** sets `merge_key` to `NULL` -> Intentionally fails the match condition and falls through to `WHEN NOT MATCHED`, executing an **`INSERT` for the new active record**!

| SCD Strategy | MERGE INTO Execution Logic | Historical State |
| :--- | :--- | :--- |
| **SCD Type 1** | `WHEN MATCHED THEN UPDATE SET target.col = src.col` | None (Overwrite) |
| **SCD Type 2** | `UNION ALL` (1 match for update + 1 NULL match for insert) | Complete (Full History) |

### 🧩 Hands-On Task 2.2: Apply Promotion Updates via `MERGE INTO` (SCD Type 2)

> **Scenario**: Elena Rossi (`C001`) was promoted to **Lead Data Engineer** on `2026-09-15`.
Complete the Delta `MERGE INTO` query to expire her old record and insert the updated active row.

Write a Delta Lake MERGE INTO statement to:

- Deactivate her existing row (is_current = FALSE, valid_to = '2026-09-15').
- Insert a new active row (is_current = TRUE, valid_from = '2026-09-15', valid_to = NULL).

In [0]:
# 2.2.1 Stage incoming promotion event
updates_df = spark.createDataFrame([
    ("C001", "Elena Rossi", "Data Engineering", "Lead", "2026-09-15")
], ["consultant_id", "full_name", "practice", "seniority_level", "effective_date"])

updates_df.createOrReplaceTempView("consultant_updates")

# TASK 2.2.2: Complete the MERGE query flags
spark.sql(f"""
MERGE INTO `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`dim_consultant` target
USING (
    SELECT 
        consultant_id AS merge_key,
        consultant_id, full_name, practice, seniority_level, effective_date
    FROM consultant_updates
    UNION ALL
    SELECT 
        NULL AS merge_key,
        u.consultant_id, u.full_name, u.practice, u.seniority_level, u.effective_date
    FROM consultant_updates u
    JOIN `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`dim_consultant` d ON u.consultant_id = d.consultant_id
    WHERE d.is_current = TRUE AND d.seniority_level <> u.seniority_level
) src
ON target.consultant_id = src.merge_key AND target.is_current = TRUE
WHEN MATCHED AND target.seniority_level <> src.seniority_level THEN
  UPDATE SET 
    -- Task A: Expire active record
    target.is_current = ..............,
    target.valid_to = CAST(src.effective_date AS DATE)
WHEN NOT MATCHED THEN
  INSERT (
    consultant_hash_key, consultant_id, full_name, practice, seniority_level, is_current, valid_from, valid_to
  )
  VALUES (
    sha2(concat_ws('||', src.consultant_id, src.full_name, src.seniority_level), 256),
    src.consultant_id, src.full_name, src.practice, src.seniority_level, .................., CAST(src.effective_date AS DATE), NULL
  )
""")

print("✨ SCD Type 2 Merge Complete! ✨")
display(spark.sql(f"SELECT * FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`dim_consultant` ORDER BY consultant_id, valid_from"))

%md
## 📚 Theory Break #3: Automated Pipeline Execution via Databricks Jobs

Running notebook code manually is fine for workshops, but client enterprise pipelines must run reliably without human intervention.

### **Databricks Workflows & Jobs**,
Databricks **Jobs** allow you to orchestrate notebooks, SQL queries, and Python scripts into automated, scheduled workflows:,
- **Schedules & Triggers:** Run pipelines on a cron schedule (e.g., daily at 02:00 AM) or trigger them via event notifications.,
- **Job Clusters:** Databricks automatically provisions ephemeral compute clusters to run the job, terminating them immediately after completion to optimize cloud costs.
- **Automation Scripts:** Instead of editing notebooks manually, automated scripts run background batch updates to apply new dimension events.

## Step 3: Automated Job Execution & Audit Verification

Now you will simulate a production batch update!

We have provided a pre-written background script (`job_update_maya.py`) that updates **Maya Lin**(`C003`)—assigning her from practice **'Cloud Architecture'** to **'AI & Decision Intelligence'** and promoting her from **'Unassigned'** to **'Senior Consultant'** effective **September 17, 2026**.

### 🧩 Hands-On Task 3: Create and Run the Databricks Job Manually

- In the left Databricks sidebar, navigate to **Jobs&Pipelines** -> **Jobs**
- Click **Create New** -> Job in the top.
- Give a name to your job like `SCD_Update_Maya_Lin_{your_initials}`.
- Click on **Add another task type** and select Python script.
- Set **Task name** to `update_Maya`.
- On **Path** select the path to `job_update_Maya.py`.
- Click **Create**, then click **Run Now** in the top right corner!
- Wait for the job run status to show **Succeeded**.

###🔍 Verify Table Update After Job Execution

After your Databricks Job run shows **Succeeded**, you should see that Maya Lin (`C003`) now has **two rows**: her historical row marked as `is_current = FALSE` and her new row marked as `is_current = TRUE`!

> Execute the cell below to inspect `dim_consultant`.

In [0]:
# COMMAND ----------
# 3.1 Verify SCD Type 2 state for Maya Lin (C003) after Job run
display(spark.sql(f"""
    SELECT 
        consultant_id, 
        full_name, 
        practice, 
        seniority_level, 
        is_current, 
        valid_from, 
        valid_to
    FROM `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`dim_consultant` 
    WHERE consultant_id = 'C003'
    ORDER BY valid_from ASC
"""))